# Eureka · Afinar la voz de Marc (v2)

**Entorno:** Entorno de ejecución → Cambiar tipo → **GPU T4** y versión **2025.07** → Guardar.

Lo hace Claude. Marc solo tiene que pulsar ▶ en el paso 3 (conectar Drive) y **Permitir**.
Todo se guarda en Drive (`Eureka clon voz/`), así que si Colab se corta no se pierde nada.


In [ ]:
# 1. GPU
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU')


In [ ]:
# 2. Instalar (5 min)
!git clone -q https://github.com/Ahmed-Ezzat20/chatterbox-finetuning-multilingual.git cbft
%cd /content/cbft
!pip install -q --no-deps -e . 2>&1 | tail -1
!pip install -q "setuptools<81" librosa==0.11.0 s3tokenizer "onnx>=1.18.0,<1.19.0" transformers==4.46.3 diffusers==0.29.0 resemble-perth==1.0.1 conformer==0.3.2 safetensors==0.5.3 soundfile pandas matplotlib faster-whisper 2>&1 | grep -i -E "^error" | head -5
!python -c "import torch; from chatterbox.mtl_tts import ChatterboxMultilingualTTS; print('INSTALACIÓN OK · GPU:', torch.cuda.is_available())"


In [ ]:
# 3. (MARC) Conectar Drive y traer las grabaciones → pulsa ▶ y luego Permitir
from google.colab import drive; drive.mount('/content/drive')
%cd /content/cbft
!find /content/drive/MyDrive -maxdepth 2 -name "voz_marc_parte*.zip" -exec cp {} . \; && unzip -oq voz_marc_parte1.zip && unzip -oq voz_marc_parte2.zip
import pandas as pd, os; df = pd.read_csv('audio_data/metadata.csv'); print(len(df), 'frases,', df.file_name.map(lambda f: os.path.exists('audio_data/'+f)).sum(), 'audios')


In [ ]:
# 4. Ajustes: español, 10 vueltas, poca RAM, cada vuelta se guarda en Drive (sin estado del optimizador)
s = open('lora.py').read()
s = s.replace("language_id='ar'", "language_id='es'").replace('language_id: str = "ar"', 'language_id: str = "es"').replace("else 4", "else 0")
import re
for k, v in [('EPOCHS', '10'), ('WARMUP_STEPS', '100'), ('SAVE_EVERY_N_STEPS', '100000'), ('LEARNING_RATE', '5e-5'), ('CHECKPOINT_DIR', '"/content/drive/MyDrive/Eureka clon voz/checkpoints"')]:
    s = re.sub(rf'^{k} = .*$', f'{k} = {v}', s, flags=re.M)
s = s.replace("'optimizer_state_dict': optimizer.state_dict(),", "")
s = s.replace('print("Creating merged model...")', 'print("FIN"); import sys; sys.exit(0)')
open('lora.py', 'w').write(s)
print([l for l in s.splitlines() if l.startswith(('EPOCHS', 'CHECKPOINT_DIR', 'LEARNING_RATE'))])


In [ ]:
# 5. Entrenar en segundo plano (~80 min)
!nohup python lora.py > train.log 2>&1 &
!sleep 5; echo lanzado


In [ ]:
# 6. Ver progreso (Claude lo ejecuta cada pocos minutos: así Colab no se duerme)
!tail -c 3000 train.log | tr '\r' '\n' | grep -a -E 'Epoch|Val Loss|rror|Killed|FIN' | tail -4; free -h | head -2; ls '/content/drive/MyDrive/Eureka clon voz/checkpoints' 2>/dev/null


In [ ]:
# 7. Examen de cada vuelta: texto entero (Whisper), parecido a tu voz y brillo de la 's'. Audios en Drive/eval
!echo aW1wb3J0IHN5cywgZ2xvYiwgcmUsIG9zLCBudW1weSBhcyBucCwgdG9yY2gsIHNvdW5kZmlsZSBhcyBzZiwgcGFuZGFzIGFzIHBkLCBsaWJyb3NhCnN5cy5wYXRoLmluc2VydCgwLCAiLiIpCmZyb20gY2hhdHRlcmJveC5tdGxfdHRzIGltcG9ydCBDaGF0dGVyYm94TXVsdGlsaW5ndWFsVFRTCmZyb20gbG9yYSBpbXBvcnQgaW5qZWN0X2xvcmFfbGF5ZXJzCmZyb20gZmFzdGVyX3doaXNwZXIgaW1wb3J0IFdoaXNwZXJNb2RlbApmcm9tIGRpZmZsaWIgaW1wb3J0IFNlcXVlbmNlTWF0Y2hlcgpEID0gIi9jb250ZW50L2RyaXZlL015RHJpdmUvRXVyZWthIGNsb24gdm96IgpjZmcgPSB0b3JjaC5sb2FkKEQgKyAiL2ZpbmFsX2xvcmFfYWRhcHRlci5wdCIsIG1hcF9sb2NhdGlvbj0iY3B1IilbImxvcmFfY29uZmlnIl0KbSA9IENoYXR0ZXJib3hNdWx0aWxpbmd1YWxUVFMuZnJvbV9wcmV0cmFpbmVkKGRldmljZT0iY3VkYSIpCkwgPSBpbmplY3RfbG9yYV9sYXllcnMobS50My50Zm1yLCBjZmdbInRhcmdldF9tb2R1bGVzIl0sIHJhbms9Y2ZnWyJyYW5rIl0sIGFscGhhPWNmZ1siYWxwaGEiXSwgZHJvcG91dD0wLjApCmRlZiBsb2FkKHApOgogICAgZCA9IHRvcmNoLmxvYWQocCwgbWFwX2xvY2F0aW9uPSJjdWRhIikKICAgIGlmICJsb3JhX3dlaWdodHMiIGluIGQ6CiAgICAgICAgVyA9IHtrOiAodlsibG9yYV9BIl0sIHZbImxvcmFfQiJdKSBmb3IgaywgdiBpbiBkWyJsb3JhX3dlaWdodHMiXS5pdGVtcygpfQogICAgZWxzZToKICAgICAgICBzID0gZFsibG9yYV9zdGF0ZV9kaWN0Il07IFcgPSB7a1s6LTddOiAoc1trXSwgc1trWzotN10gKyAiLmxvcmFfQiJdKSBmb3IgayBpbiBzIGlmIGsuZW5kc3dpdGgoIi5sb3JhX0EiKX0KICAgIG4gPSAwCiAgICBmb3IgaywgKGEsIGIpIGluIFcuaXRlbXMoKToKICAgICAgICBpZiBrIGluIEw6IExba10ubG9yYV9BLmRhdGEgPSBhLnRvKCJjdWRhIik7IExba10ubG9yYV9CLmRhdGEgPSBiLnRvKCJjdWRhIik7IG4gKz0gMQogICAgcmV0dXJuIG4KZGYgPSBwZC5yZWFkX2NzdigiYXVkaW9fZGF0YS9tZXRhZGF0YS5jc3YiKQpxcyA9IGRmW2RmLnRyYW5zY3JpcHRpb24uc3RyLnN0cmlwKCkuc3RyLmVuZHN3aXRoKCI/IikgJiAoZGYuZHVyYXRpb25fc2Vjb25kcyA+IDIuNSkgJiAoZGYuZHVyYXRpb25fc2Vjb25kcyA8IDgpXQpSRUZRID0gImF1ZGlvX2RhdGEvIiArIHFzLmlsb2NbMF0uZmlsZV9uYW1lClJFRkEgPSAiYXVkaW9fZGF0YS8iICsgZGZbKGRmLmR1cmF0aW9uX3NlY29uZHMgPiA3KSAmIChkZi5kdXJhdGlvbl9zZWNvbmRzIDwgMTApXS5pbG9jWzBdLmZpbGVfbmFtZQpyZWFsID0gW2xpYnJvc2EubG9hZCgiYXVkaW9fZGF0YS8iICsgZiwgc3I9MTYwMDApWzBdIGZvciBmIGluIGRmLnNhbXBsZSgyNSwgcmFuZG9tX3N0YXRlPTEpLmZpbGVfbmFtZV0Kc3BrID0gbS52ZS5lbWJlZHNfZnJvbV93YXZzKHJlYWwsIDE2MDAwLCBhc19zcGs9VHJ1ZSkKY2VuX3JlYWwgPSBucC5tZWFuKFtsaWJyb3NhLmZlYXR1cmUuc3BlY3RyYWxfY2VudHJvaWQoeT13LCBzcj0xNjAwMCkubWVhbigpIGZvciB3IGluIHJlYWxdKQphc3IgPSBXaGlzcGVyTW9kZWwoInNtYWxsIiwgZGV2aWNlPSJjdWRhIiwgY29tcHV0ZV90eXBlPSJmbG9hdDE2IikKbm9ybSA9IGxhbWJkYSB0OiByZS5zdWIociJbXmEtesOhw6nDrcOzw7rDscO8XSIsICIiLCB0Lmxvd2VyKCkpCmRlZiBnZW4odCwgbj0xKToKICAgIHEgPSB0LnN0cmlwKCkuZW5kc3dpdGgoIj8iKQogICAgYmVzdCA9IE5vbmUKICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgIHcgPSBtLmdlbmVyYXRlKHQsIGxhbmd1YWdlX2lkPSJlcyIsIGF1ZGlvX3Byb21wdF9wYXRoPVJFRlEgaWYgcSBlbHNlIFJFRkEsIGV4YWdnZXJhdGlvbj0wLjcgaWYgcSBlbHNlIDAuNiwgY2ZnX3dlaWdodD0wLjUsIHRlbXBlcmF0dXJlPTAuNikuc3F1ZWV6ZSgpLmNwdSgpLm51bXB5KCkKICAgICAgICB3MTYgPSBsaWJyb3NhLnJlc2FtcGxlKHcsIG9yaWdfc3I9bS5zciwgdGFyZ2V0X3NyPTE2MDAwKQogICAgICAgIGh5cCA9ICIgIi5qb2luKHMudGV4dCBmb3IgcyBpbiBhc3IudHJhbnNjcmliZSh3MTYsIGxhbmd1YWdlPSJlcyIsIGJlYW1fc2l6ZT01KVswXSkKICAgICAgICBzaW0gPSBTZXF1ZW5jZU1hdGNoZXIoTm9uZSwgbm9ybSh0KSwgbm9ybShoeXApKS5yYXRpbygpCiAgICAgICAgaWYgYmVzdCBpcyBOb25lIG9yIHNpbSA+IGJlc3RbMV06IGJlc3QgPSAodywgc2ltLCBoeXAsIHcxNikKICAgICAgICBpZiBzaW0gPiAwLjk3OiBicmVhawogICAgcmV0dXJuIGJlc3QKVCA9IFsiwr9TYWLDrWFzIHF1ZSBlbCBwdWxwbyB0aWVuZSB0cmVzIGNvcmF6b25lcz8iLCAiRG9zIHNlIGVuY2FyZ2FuIGRlIGxhcyBicmFucXVpYXMsIHkgZWwgdGVyY2VybywgZGVsIHJlc3RvIGRlbCBjdWVycG8uIiwgIlkgbG8gbcOhcyBjdXJpb3NvIGVzIHF1ZSwgY3VhbmRvIG5hZGEsIGVsIGNvcmF6w7NuIHByaW5jaXBhbCBzZSBwYXJhLiIsICLCv1RlIGltYWdpbmFzIHNvYnJldml2aXIgYWwgVGl0YW5pYz8iLCAiTGEgU2FncmFkYSBGYW3DrWxpYSB5YSBlcyBsYSBpZ2xlc2lhIG3DoXMgYWx0YSBkZWwgbXVuZG8uIiwgIlPDrWd1ZW1lLCB5IG3DoW5kYXNlbG8gYSBlc2UgYW1pZ28gcXVlIHNpZW1wcmUgbGxlZ2EgdGFyZGUuIl0KZGVmIGV2YWx1YXRlKG5hbWUsIG49MSwgc2F2ZT1UcnVlKToKICAgIFIgPSBbZ2VuKHQsIG4pIGZvciB0IGluIFRdCiAgICBzaW1zID0gW3JbMV0gZm9yIHIgaW4gUl0KICAgIGVtYiA9IG0udmUuZW1iZWRzX2Zyb21fd2F2cyhbclszXSBmb3IgciBpbiBSXSwgMTYwMDAsIGFzX3Nwaz1UcnVlKQogICAgc3BrX3NpbSA9IGZsb2F0KG5wLmRvdChlbWIsIHNwaykgLyAobnAubGluYWxnLm5vcm0oZW1iKSAqIG5wLmxpbmFsZy5ub3JtKHNwaykpKQogICAgY2VuID0gbnAubWVhbihbbGlicm9zYS5mZWF0dXJlLnNwZWN0cmFsX2NlbnRyb2lkKHk9clszXSwgc3I9MTYwMDApLm1lYW4oKSBmb3IgciBpbiBSXSkKICAgIGlmIHNhdmU6CiAgICAgICAgc2lsID0gbnAuemVyb3MoaW50KDAuOCAqIG0uc3IpKTsgb3MubWFrZWRpcnMoRCArICIvZXZhbCIsIGV4aXN0X29rPVRydWUpOyBzZi53cml0ZShmIntEfS9ldmFsL3tuYW1lfS53YXYiLCBucC5jb25jYXRlbmF0ZShbbnAuY29uY2F0ZW5hdGUoW3JbMF0sIHNpbF0pIGZvciByIGluIFJdKSwgbS5zcikKICAgIHByaW50KGYie25hbWU6MzJzfSB0ZXh0byB7bnAubWVhbihzaW1zKTouM2Z9IG1pbiB7bWluKHNpbXMpOi4zZn0gfCBwYXJlY2lkbyB2b3oge3Nwa19zaW06LjNmfSB8IGNlbnRyb2lkZSB7Y2VuOi4wZn0gKHJlYWwge2Nlbl9yZWFsOi4wZn0pIiwgZmx1c2g9VHJ1ZSkKICAgIGZvciB0LCByIGluIHppcChULCBSKToKICAgICAgICBpZiByWzFdIDwgMC45NTogcHJpbnQoIiAgICIsIHJvdW5kKHJbMV0sIDIpLCAifCIsIHJbMl0pCiAgICByZXR1cm4gbnAubWVhbihzaW1zKSwgc3BrX3NpbQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgUCA9IHN5cy5hcmd2WzE6XSBvciBbRCArICIvZmluYWxfbG9yYV9hZGFwdGVyLnB0Il0gKyBzb3J0ZWQoZ2xvYi5nbG9iKEQgKyAiL2NoZWNrcG9pbnRzL2NoZWNrcG9pbnRfZXBvY2gqLnB0IiksIGtleT1sYW1iZGEgcDogaW50KHJlLnNlYXJjaChyInN0ZXAoXGQrKSIsIHApLmdyb3VwKDEpKSkgKyBbRCArICIvY2hlY2twb2ludHMvZmluYWxfbG9yYV9hZGFwdGVyLnB0Il0KICAgIGZvciBwIGluIFA6CiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMocCk6IGxvYWQocCk7IGV2YWx1YXRlKG9zLnBhdGguYmFzZW5hbWUob3MucGF0aC5kaXJuYW1lKHApKSArICJfIiArIG9zLnBhdGguYmFzZW5hbWUocCkucmVwbGFjZSgiLnB0IiwgIiIpKQo= | base64 -d > evalc.py
!python evalc.py 2>&1 | grep -v -i -E 'warn|Sampling|Fetching|tensorflow|cuda_|E0000|absl|rebuild|AVX' | tail -60
